# Chapter 11. 시퀀스 모델 — 실습 노트북: RNN 은닉 상태와 순전파

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter11_1_rnn_forward.ipynb)

책 본문: [11.1 RNN: 은닉 상태와 순전파](https://smhanlab.com/book-ml/kor/ml1/chapter11/1.html)

이 노트북은 책의 수치 예제(스칼라 RNN 3단계 손계산, 파라미터 수 514, 고정점 0.8196,
누적합 예제)를 코드 그대로 실행하고, 수렴 곡선을 그립니다. numpy/matplotlib만 씁니다.

## 1. 스칼라 RNN: 본문 "손으로 한 번" 예제 그대로

$h_t = \tanh(w_{xh} x_t + w_{hh} h_{t-1} + b_h)$, $w_{xh}=0.5$, $w_{hh}=0.8$, $b_h=0$,
$x_t=1.0$ 고정, $h_0=0$. 본문의 손계산 값(0.462, 0.701, 0.786)과 일치하는지 확인합니다.

In [1]:
import math

def rnn_step(x, h, wxh, whh, b=0.0):
    return math.tanh(wxh * x + whh * h + b)

def rnn_forward_scalar(inputs, h0, wxh, whh, b=0.0):
    hs, h = [], h0
    for x in inputs:
        h = rnn_step(x, h, wxh, whh, b)
        hs.append(h)
    return hs

hs = rnn_forward_scalar([1.0, 1.0, 1.0], h0=0.0, wxh=0.5, whh=0.8)
for i, v in enumerate(hs, 1):
    print(f"h{i} = {v:.4f}")

hand = [0.462, 0.701, 0.786]   # 본문 손계산
assert all(abs(a - b) < 5e-4 for a, b in zip(hs, hand)), "본문 값과 불일치!"
print("본문 손계산 값과 일치: OK")

h1 = 0.4621
h2 = 0.7012
h3 = 0.7860
본문 손계산 값과 일치: OK


## 2. 파라미터 수: 시퀀스 길이와 무관한 514

11.2절 실습 모델(어휘 $V=26$, 은닉 $H=8$)의 파라미터를 세어봅니다.
문장이 10글자든 1000글자든 이 숫자는 **514**로 고정 — 파라미터 공유의 직접적 혜택입니다.

In [2]:
H, V = 8, 26
params = [("W_xh (입력→은닉)", H * V), ("W_hh (은닉→은닉)", H * H),
          ("W_hy (은닉→출력)", V * H), ("b_h", H), ("b_y", V)]
total = 0
for name, n in params:
    total += n
    print(f"{name:16s}: {n}")
print(f"{'합계':16s}: {total}")
print(f"비교 — MLP 5글자 윈도우: 첫 층 가중치 {5*26*8+8}개 (RNN 전체의 { (5*26*8+8)/total:.1f}배)")

W_xh (입력→은닉)    : 208
W_hh (은닉→은닉)    : 64
W_hy (은닉→출력)    : 208
b_h             : 8
b_y             : 26
합계              : 514
비교 — MLP 5글자 윈도우: 첫 층 가중치 1048개 (RNN 전체의 2.0배)


## 3. 수렴: 고정점, 수렴 속도, 그리고 11.3절의 연결

같은 입력이 계속 오면 $h_t$는 고정점 $h^* = \tanh(0.5 + w_{hh} h^*)$로 수렴합니다.
수렴 속도는 고정점 근처의 증폭 인자 $w_{hh}(1-(h^*)^2)$가 결정하는데, 이 곱은 11.2절의
$\tanh'(z_t) W_{hh}$ — 즉 **BPTT의 그래디언트 소실 인자와 동일한 형태**입니다.
11.3절의 그래디언트 소실을 순전파에서 미리 체감하는 순간입니다.

In [3]:
def fixed_point(wxh, whh, b=0.0, x=1.0, iters=500):
    h = 0.0
    for _ in range(iters):
        h = math.tanh(wxh * x + whh * h + b)
    return h

for whh in (0.8, 0.9):
    f = fixed_point(0.5, whh)
    amp = whh * (1 - f * f)
    h, steps = 0.0, 0
    while abs(h - f) > 1e-3:
        h = math.tanh(0.5 + whh * h)
        steps += 1
    print(f"w_hh={whh}: 고정점 h* = {f:.4f}, 증폭 인자 = {amp:.4f}, 1e-3 도달: {steps}단계")

w_hh=0.8: 고정점 h* = 0.8196, 증폭 인자 = 0.2626, 1e-3 도달: 6단계
w_hh=0.9: 고정점 h* = 0.8532, 증폭 인자 = 0.2448, 1e-3 도달: 6단계


In [4]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

for _f in ["Noto Sans CJK KR", "NanumGothic", "Malgun Gothic", "AppleGothic"]:
    if any(_f.lower() == x.name.lower() for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["svg.fonttype"] = "path"

fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=120)
for whh, c, ls in [(0.8, "#1971c2", "-"), (0.9, "#e8590c", "--")]:
    f = fixed_point(0.5, whh)
    h, ts, hs_ = 0.0, [0.0], [0.0]
    for t in range(1, 31):
        h = math.tanh(0.5 + whh * h)
        ts.append(t); hs_.append(h)
    ax.plot(ts, hs_, color=c, ls=ls, lw=2, label=f"$w_{{hh}}$={whh}")
    ax.axhline(f, color=c, lw=0.8, alpha=0.4)
    ax.scatter([1, 2, 3], hs_[:3], color=c, zorder=3)
    ax.annotate(f"$h^*$≈{f:.3f}", (ts[-1], f),
                textcoords="offset points", xytext=(-52, 6), fontsize=9, color=c)
ax.set_xlabel("시점 t"); ax.set_ylabel("$h_t$")
ax.set_title("스칼라 RNN 은닉 상태의 수렴: $h_t = \\tanh(0.5 + w_{hh} h_{t-1})$, $x_t=1$")
ax.set_xlim(0, 30); ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.tight_layout()
IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
fig.savefig(f"{IMG}/ch11_1_scalar_convergence.svg")
print("수렴 곡선 저장 완료 — 두 곡선 모두 약 6단계 만에 고정점에 정착")

수렴 곡선 저장 완료 — 두 곡선 모두 약 6단계 만에 고정점에 정착


## 4. 은닉 상태가 '요약'을 실제로 수행하는 모습: 누적합

$w_{xh}=0.1$, $w_{hh}=0.9$인 스칼라 RNN에 $[2,3,1,4]$를 넣으면, $\tanh$ 전의
$z_t$는 **과거 입력들의 기하급수적으로 감쇠된 가중합** $z_t \approx 0.1(x_t + 0.9 x_{t-1}
+ 0.9^2 x_{t-2} + \cdots)$에 가깝습니다 — $0.9^k$가 $k$칸 전 입력의 '기억 세기'.
$10 h_t$가 누적합을 추정하는 데 쓰이면, 초기엔 거의 정확하다가 $\tanh$ 포화로
점점 아래로 벗어납니다 — '요약 벡터가 $[-1,1]$에 갇힌다'는 한계의 직접적 모습.

In [5]:
h, cumsum = 0.0, 0.0
for x in [2.0, 3.0, 1.0, 4.0]:
    z = 0.1 * x + 0.9 * h
    h = math.tanh(z)
    cumsum += x
    print(f"x={x}  z={z:.4f}  h={h:.4f}  누적합={cumsum}  추정치 10h={10*h:.2f}")

print(f"\n기억 감쇠: 0.9^10 = {0.9**10:.3f}, 0.9^20 = {0.9**20:.3f}, 0.5^10 = {0.5**10:.4f}")
print("w_hh를 0.5로 하면 10단어 전 기억의 0.1%만 남는다 — '기억의 길이'는 w_hh 하나로 조절된다")

x=2.0  z=0.2000  h=0.1974  누적합=2.0  추정치 10h=1.97
x=3.0  z=0.4776  h=0.4443  누적합=5.0  추정치 10h=4.44
x=1.0  z=0.4999  h=0.4621  누적합=6.0  추정치 10h=4.62
x=4.0  z=0.8158  h=0.6728  누적합=10.0  추정치 10h=6.73

기억 감쇠: 0.9^10 = 0.349, 0.9^20 = 0.122, 0.5^10 = 0.0010
w_hh를 0.5로 하면 10단어 전 기억의 0.1%만 남는다 — '기억의 길이'는 w_hh 하나로 조절된다


## 5. 한 줄 요약

- **은닉 상태 $h_t$는 과거 전체 $x_1, \ldots, x_t$의 함수** — $W_{hh} h_{t-1}$ 한 항에 전부 달려 있다.
- **파라미터($W_{xh}, W_{hh}, W_{hy}$)는 시점 전체에서 공유** — 시퀀스 길이에 무관(514개 고정).
- **수렴 속도 = 증폭 인자 $w_{hh}(1-(h^*)^2)$** — 11.3절 그래디언트 소실과 같은 수식.
- 다음 노트북(11.2)에서는 이 구조를 학습(BPTT)시켜 문자 단위 언어모델을 만들어봅니다.